In [1]:
from datasets import load_dataset
from LLMGeometry.datasets import CACHE_DIR
import numpy as np
import pandas as pd
import pickle
raw_dataset = load_dataset("fancyzhx/ag_news", trust_remote_code=True, cache_dir=CACHE_DIR) # Loading the dataset

In [2]:
def process_dataset(df, random_state=42):

    if 'word_count' not in df.columns:
        df['word_count'] = df['text'].apply(lambda x: len(x.split())) # Counting the number of words in each text
        df = df.query('word_count <= 24').copy() # Removing long texts

    # For remaining labels subselecting so that they are equally represented (to avoid class imbalance). Find the smallest class size and subselect all classes to that size
    min_class_size = df['label'].value_counts().min()
    df = df.groupby('label').apply(lambda x: x.sample(min_class_size, random_state=random_state)).reset_index(drop=True)

    # Renaming columns
    df = df.rename(columns={'label': 'label', 'text': 'text'})

    # Mapping labels
    mapping = {
        0: 'World',
        1: 'Sports',
        2: 'Business',
        3: 'Sci/Tech'
    }

    letter_mapping = {
        0: 'A',
        1: 'B',
        2: 'C',
        3: 'D'
    }
    
    df['category'] = df['label'].map(mapping)
    df['category'] = df['category'].str.title()
    df['category_letter'] = df['label'].map(letter_mapping) 

    # Shuffling
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return df

In [ ]:
train_df = raw_dataset['train'].to_pandas()
test_df = raw_dataset['test'].to_pandas()

# Counting words
train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))
test_df['word_count'] = test_df['text'].apply(lambda x: len(x.split()))

# Selecting only short texts
train_df = train_df.query('word_count <= 24').copy()[:2000]
test_df = test_df.query('word_count <= 24').copy()

# Processing the datasets
train_df = process_dataset(train_df)
test_df = process_dataset(test_df)

In [ ]:
# Shuffled mapping
shuffle_map = {
    'World' : 'Sports',
    'Sports' : 'Business',
    'Business' : 'Sci/Tech',
    'Sci/Tech' : 'World'
}

train_df['category_shuffle'] = train_df['category'].map(shuffle_map)
test_df['category_shuffle'] = test_df['category'].map(shuffle_map)

# These rows contain URLs
train_df = train_df.drop(213)
train_df = train_df.drop(209) 
train_df = train_df.reset_index(drop=True)

In [55]:
with open('ag_news.pickle', 'wb') as f:
    pd.to_pickle({'train': train_df, 'test': test_df}, f)